[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VectorInstitute/synthetic-data-bootcamp/blob/main/implementations/qa_text_generation/03_quality_filtering.ipynb)

# Step 3 — Quality Filtering and LLM-as-Judge

Filter synthetic Q&A with cheap heuristics, then score survivors with a judge model.

## Learning objectives
- Apply deduplication, format checks, and prompt-leakage detection
- Score samples on correctness, coherence, instruction-following, and plausibility
- Optionally compare candidates pairwise against seed examples

In [1]:
from pathlib import Path

from aieng.syn_data.text import (
    DEFAULT_JUDGE_THRESHOLD,
    RESULTS_DIR,
    SYNTHETIC_FILTERED_PATH,
    SYNTHETIC_RAW_PATH,
    QASample,
    apply_heuristic_filters,
    create_judge_client,
    filter_with_judge,
    load_typed_jsonl,
    save_typed_jsonl,
    summarize_judge_scores,
    use_repo_root,
    write_json,
)
from dotenv import load_dotenv


load_dotenv()
use_repo_root(Path("."))

2026-06-11 15:26:48,126 INFO root: AI Engineering synthetic data utilities 

 Logging configured.


PosixPath('/Users/royajavadi/projects/synthetic-data-bootcamp')

In [2]:
# TODO: remove this before merging into main
%load_ext autoreload
%autoreload 2

## 1. Heuristic filtering

In [3]:
raw_samples = load_typed_jsonl(SYNTHETIC_RAW_PATH, QASample.from_dict)
kept, rejected = apply_heuristic_filters(raw_samples)
print(f"Kept {len(kept)} / {len(raw_samples)} samples")

Kept 19 / 20 samples


## 2. LLM-as-judge absolute scoring

Here the judge model is evaluating quality of the generated data by our teacher model after applying basic heuristic filtering. The minimum pass score is defined in configs. 

In [4]:
judge = create_judge_client()

filtered_samples, judge_scores, rejected = filter_with_judge(
    judge,
    kept,
    threshold=DEFAULT_JUDGE_THRESHOLD,
)
print(
    f"After judge filter: {len(filtered_samples)} kept, {len(rejected)} rejected (heuristics + judge)",
)
summarize_judge_scores(judge_scores)

After judge filter: 19 kept, 0 rejected (heuristics + judge)


{'correctness': 5.0,
 'coherence': 5.0,
 'instruction_following': 5.0,
 'factual_plausibility': 5.0,
 'average': 5.0}

## 3. Save filtered corpus and quality report

In [5]:
save_typed_jsonl(
    SYNTHETIC_FILTERED_PATH,
    filtered_samples,
    to_dict=QASample.to_dict,
)

quality_report = {
    "input_count": len(raw_samples),
    "after_heuristics": len(kept),
    "after_judge": len(filtered_samples),
    "judge_threshold": DEFAULT_JUDGE_THRESHOLD,
    "judge_summary": summarize_judge_scores(judge_scores),
    "rejected": rejected,
}
write_json(RESULTS_DIR / "quality_report.json", quality_report)
quality_report

{'input_count': 20,
 'after_heuristics': 19,
 'after_judge': 19,
 'judge_threshold': 3.5,
 'judge_summary': {'correctness': 5.0,
  'coherence': 5.0,
  'instruction_following': 5.0,
  'factual_plausibility': 5.0,
  'average': 5.0},
 'rejected': []}